In [57]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import HeteroData
from torch_geometric.nn import MessagePassing, global_mean_pool

from models_graph import HarmonicGraphEncoder
from models_FiLMatt import AttnFiLMSEModel

import GridMLM_tokenizers
from GridMLM_tokenizers import CSGridMLMTokenizer

import pickle

import numpy as np

from generate_utils import load_AttnFiLMSEModel, nucleus_token_by_token_generate

In [58]:
# load the dataset and the tokenizer
with open("data/gjt_train.pkl", "rb") as f:
    gjt_train = pickle.load(f)

tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

In [59]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

graph_model = HarmonicGraphEncoder()
graph_model.to(device)

# load the model
model = load_AttnFiLMSEModel(
    tokenizer=tokenizer,
    guidance_dim=graph_model.output_dim
)

In [60]:
d = gjt_train[0]
print(d.keys())

dict_keys(['harmony_ids', 'attention_mask', 'pianoroll', 'time_signature', 'h_density_complexity', 'graph_ready_object'])


In [61]:
bar_start, bar_end = d['graph_ready_object'].get_valid_bar_segment_range(4)
print(bar_start, bar_end)

9 12


In [62]:
d['graph_ready_object'].make_graph_of_segment(bar_start, bar_end)
print(d['graph_ready_object'].segment_graph)

HeteroData(
  pitch={ x=[12, 12] },
  event={ x=[5, 1] },
  (pitch, participates, event)={
    edge_index=[2, 27],
    edge_attr=[27, 8],
  },
  (event, next, event)={
    edge_index=[2, 4],
    edge_attr=[4, 7],
  }
)


In [63]:
print(d['graph_ready_object'].segment_bar_objects)

[<graph_utils.Bar object at 0x7d69f82b0090>, <graph_utils.Bar object at 0x7d69f82b11d0>, <graph_utils.Bar object at 0x7d69f850a110>]


In [64]:
token_positions = d['graph_ready_object'].get_token_positions_of_bar_segment()
print(token_positions)

[46, 47, 48, 49, 51, 52, 53, 54, 56, 57, 58, 59]


In [65]:
# mask the tokens in the segment
masked_tokens = np.array(d['harmony_ids'])
masked_tokens[token_positions] = tokenizer.mask_token_id
masked_tokens = masked_tokens.tolist()      
print(masked_tokens)

[6, 276, 276, 194, 194, 6, 339, 339, 148, 148, 6, 276, 276, 71, 71, 6, 216, 216, 13, 13, 6, 159, 159, 159, 159, 6, 332, 332, 129, 129, 6, 274, 274, 71, 71, 6, 216, 216, 13, 13, 6, 159, 159, 332, 332, 6, 5, 5, 5, 5, 6, 5, 5, 5, 5, 6, 5, 5, 5, 5, 6, 270, 270, 270, 270, 6, 73, 73, 245, 245, 6, 227, 227, 226, 226, 6, 14, 14, 151, 151]


In [66]:
guidance_vector = graph_model(d['graph_ready_object'].segment_graph.to(device))
print(guidance_vector.shape)

torch.Size([128])


In [67]:
melody_grid = torch.tensor(d['pianoroll'], dtype=torch.float32).unsqueeze(0)
harmony_ids = torch.tensor(d['harmony_ids'], dtype=torch.long).unsqueeze(0)
masked_tokens_tensor = torch.tensor(masked_tokens, dtype=torch.long).unsqueeze(0)

In [68]:
new_harmony_ids = nucleus_token_by_token_generate(
    model=model,
    melody_grid=melody_grid.to(device),
    guidance_vector=None,
    mask_token_id=tokenizer.mask_token_id,
    chord_constraints=masked_tokens_tensor.to(device),
    pad_token_id=tokenizer.pad_token_id,
    nc_token_id=tokenizer.nc_token_id,
    temperature=3.0,
)

In [69]:
print(harmony_ids)
print(masked_tokens_tensor)
print(new_harmony_ids)

tensor([[  6, 276, 276, 194, 194,   6, 339, 339, 148, 148,   6, 276, 276,  71,
          71,   6, 216, 216,  13,  13,   6, 159, 159, 159, 159,   6, 332, 332,
         129, 129,   6, 274, 274,  71,  71,   6, 216, 216,  13,  13,   6, 159,
         159, 332, 332,   6, 339, 339, 148, 148,   6, 279, 279, 279, 279,   6,
         158, 158, 148, 148,   6, 270, 270, 270, 270,   6,  73,  73, 245, 245,
           6, 227, 227, 226, 226,   6,  14,  14, 151, 151]])
tensor([[  6, 276, 276, 194, 194,   6, 339, 339, 148, 148,   6, 276, 276,  71,
          71,   6, 216, 216,  13,  13,   6, 159, 159, 159, 159,   6, 332, 332,
         129, 129,   6, 274, 274,  71,  71,   6, 216, 216,  13,  13,   6, 159,
         159, 332, 332,   6,   5,   5,   5,   5,   6,   5,   5,   5,   5,   6,
           5,   5,   5,   5,   6, 270, 270, 270, 270,   6,  73,  73, 245, 245,
           6, 227, 227, 226, 226,   6,  14,  14, 151, 151]])
(tensor([[  6, 276, 276, 194, 194,   6, 339, 339, 148, 148,   6, 276, 276,  71,
        